In [ ]:
from libraries import *
from parameters import *
from util import *


In [ ]:
adata = sc.read_h5ad("./../../Data/ComboScreen.h5ad")


In [ ]:
adata_control_day4 = adata[(adata.obs["NTC"] ==1) & (adata.obs["N_genes_targeted"] ==1) & (adata.obs["time_point"] =="day04"),:]
pd.crosstab(adata_control_day4.obs["N_genes_targeted"], adata_control_day4.obs["final_label"])

In [ ]:
adata_control_day10 = adata[(adata.obs["NTC"] ==1) & (adata.obs["N_genes_targeted"] ==1) & (adata.obs["time_point"] =="day10"),:]
pd.crosstab(adata_control_day10.obs["N_genes_targeted"], adata_control_day10.obs["final_label"])

In [ ]:
adata.obs["N_genes_targeted"].value_counts()

In [ ]:
a=pd.read_csv("./../../Data/combined_sgRNA_assignment_df.final.csv", index_col=0)
a = a[a["cell"].isin(adata.obs_names)].copy()


In [ ]:
sum(a["UMI_counts"] < 30)

In [ ]:
a

In [ ]:
plt.hist(a["UMI_counts"], bins=50)
plt.yscale("log")
plt.xlabel("guide UMI counts")
plt.ylabel("Number of observations (log)")
plt.title("Distribution of guide UMI counts")
plt.show()

In [ ]:
mat = (
    a
    .pivot_table(
        index="cell",
        columns="gRNA",
        values="UMI_counts",
        aggfunc="sum",    # sum UMIs if multiple rows per cell–gRNA
        fill_value=0
    )
)

In [ ]:
mat[mat>50]=50

In [ ]:
# mat: rows=cells, columns=gRNAs, values=UMI counts
row_sums = mat.sum(axis=1)

mat_pct = mat.div(row_sums, axis=0) * 100


In [ ]:
mat_pct[mat_pct < 10] = 0

In [ ]:
mat_pct.columns

In [ ]:
gene_names = mat_pct.columns.str.rsplit("_", n=1).str[0]

# Sum columns that share the same gene name
mat_pct_gene_sum = mat_pct.groupby(gene_names, axis=1).sum()

In [ ]:
mat_pct_gene_sum[mat_pct_gene_sum>0]=1

In [ ]:
mat_pct_gene_sum.sum(axis=1).value_counts()

In [ ]:
mat_pct_gene_sum = mat_pct_gene_sum.loc[adata.obs.index,:]

In [ ]:
mat_pct_gene_sum["N_genes_targeted_new"] = mat_pct_gene_sum.sum(axis=1)

In [ ]:
mat_pct_gene_sum["final_label"] = adata.obs["final_label"]
mat_pct_gene_sum["time_point"] = adata.obs["time_point"]

In [ ]:
sum(mat_pct_gene_sum["time_point"] =="day04")

In [ ]:
new_control_day4 = mat_pct_gene_sum.loc[(mat_pct_gene_sum["NTC"] ==1) & (mat_pct_gene_sum["N_genes_targeted_new"] ==1) & (mat_pct_gene_sum["time_point"] =="day04"),:]
pd.crosstab(new_control_day4["N_genes_targeted_new"], new_control_day4["final_label"])

In [ ]:
n_nonzero_per_cell.value_counts()

In [ ]:
n_nonzero_per_cell.value_counts()

In [ ]:
onehot = pd.crosstab(a["cell"], a["gRNA"])
#onehot = (onehot > 0).astype("int8")   # one-hot, not counts

# align to adata order and fill missing cells with 0
onehot = onehot.reindex(adata.obs_names, fill_value=0)

# add to obs
#adata.obs = adata.obs.join(onehot)


In [ ]:
np.max(onehot)

In [ ]:
adata_control=adata[adata.obs[['NTC_1', 'NTC_2', 'NTC_3',
       'NTC_4', 'NTC_5']].sum(axis=1)==1,:]

In [ ]:
def combine_onehot(row):
    # Select all column names where value == 1
    active = [col for col in ['NTC_1', 'NTC_2', 'NTC_3',
       'NTC_4', 'NTC_5'] if row[col] == 1]
    # Join multiple actives with '+', or return 'None' if none are active
    return '+'.join(active) if active else 'None'

adata_control.obs['controlGuide'] = adata_control.obs[['NTC_1', 'NTC_2', 'NTC_3',
       'NTC_4', 'NTC_5']].apply(combine_onehot, axis=1)


In [ ]:
sc.pp.normalize_total(adata_control, target_sum=20000)
sc.pp.log1p(adata_control)


In [ ]:
sc.pp.highly_variable_genes(adata_control, n_top_genes=3000)

In [ ]:
sc.pp.scale(adata_control, max_value=9)
sc.pp.pca(adata_control, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(adata_control, 
                n_neighbors=8,
                metric=par_downstream_neighbor_metric,
                n_pcs=50)
sc.tl.umap(adata_control)


In [ ]:
sc.pl.umap(adata_control, color='final_label', 
           #legend_loc='on data', 
           legend_fontoutline=3, 
           legend_fontsize=14, 
           legend_fontweight='normal', 
           show=False, 
           size=0.3)

In [ ]:
adata_control.obs[[]]